In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/calibrated_pressure_predictions.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/events_with_spatial_temporal_candidates.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_lookup_250m.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/directed_candidate_graph_1000m.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_time_model_table.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/zone_observation_confidence.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-p6/local_pressure_explanations.parquet


# Phase 6

## Operational Risk Components

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy.sparse import csr_matrix

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

OUTPUT_DIR = Path("/kaggle/working/parking_phase6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input_file(filename):
    matches = list(Path("/kaggle/input").rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found under /kaggle/input."
        )

    if len(matches) > 1:
        print(f"Using {matches[0]}")

    return matches[0]


MODEL_TABLE_PATH = find_input_file(
    "zone_time_model_table.parquet"
)

PREDICTIONS_PATH = find_input_file(
    "calibrated_pressure_predictions.parquet"
)

ZONE_LOOKUP_PATH = find_input_file(
    "zone_lookup_250m.parquet"
)

CANDIDATE_GRAPH_PATH = find_input_file(
    "directed_candidate_graph_1000m.parquet"
)

CONFIDENCE_PATH = find_input_file(
    "zone_observation_confidence.csv"
)

EVENTS_PATH = find_input_file(
    "events_with_spatial_temporal_candidates.parquet"
)

EXPLANATIONS_PATH = find_input_file(
    "local_pressure_explanations.parquet"
)

model_columns = pq.ParquetFile(
    MODEL_TABLE_PATH
).schema.names

context_candidates = [
    "incident_count",
    "unique_vehicles",
    "main_road_events",
    "road_crossing_events",
    "footpath_events",
    "named_junction_events",
    "large_vehicle_events",
    "multi_offence_events",
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share",
    "current_burst_z",
    "relative_to_same_hour",
    "short_term_trend",
    "consecutive_active_hours_before",
    "observation_confidence",
    "observer_bias_penalty",
    "duplicate_capture_rate",
    "cold_start_zone"
]

available_context_columns = [
    col for col in context_candidates
    if col in model_columns
]

print("Available contextual columns:")
print(available_context_columns)

Available contextual columns:
['incident_count', 'unique_vehicles', 'main_road_events', 'road_crossing_events', 'footpath_events', 'named_junction_events', 'large_vehicle_events', 'multi_offence_events', 'main_road_share', 'road_crossing_share', 'footpath_share', 'named_junction_share', 'large_vehicle_share', 'multi_offence_share', 'current_burst_z', 'relative_to_same_hour', 'short_term_trend', 'consecutive_active_hours_before', 'observation_confidence', 'observer_bias_penalty', 'duplicate_capture_rate', 'cold_start_zone']


In [3]:
# Build the aligned operational table

# Each row represents information available during hour t and a forecast for hour t+1. 
# Predictions are merged using zone_index and target_time,
required_panel_columns = [
    "zone_index",
    "time_window",
    "target_time",
    "split_code",
    "target_next_hour_pressure"
]

panel_columns = list(dict.fromkeys(
    required_panel_columns
    + available_context_columns
))

operational_panel = pd.read_parquet(
    MODEL_TABLE_PATH,
    columns=panel_columns,
    filters=[
        ("split_code", "in", [2, 3])
    ]
)

prediction_columns = [
    "zone_index",
    "target_time",
    "actual_pressure",
    "predicted_pressure",
    "calibrated_pressure",
    "hotspot_probability"
]

calibrated_predictions = pd.read_parquet(
    PREDICTIONS_PATH,
    columns=prediction_columns
)

operational_panel = operational_panel.merge(
    calibrated_predictions,
    on=["zone_index", "target_time"],
    how="inner",
    validate="one_to_one"
)

operational_panel = (
    operational_panel
    .sort_values(["time_window", "zone_index"])
    .reset_index(drop=True)
)

assert np.allclose(
    operational_panel[
        "target_next_hour_pressure"
    ],
    operational_panel["actual_pressure"]
)

print("Operational rows:", len(operational_panel))
print(
    "Hours:",
    operational_panel["time_window"].nunique()
)
print(
    "Zones:",
    operational_panel["zone_index"].nunique()
)
print(
    "Memory MB:",
    operational_panel.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Operational rows: 1265344
Hours: 1088
Zones: 1163
Memory MB: 149.63415908813477


In [4]:
# Recreate validated distance-based exposure

# This applies inverse-distance weighting to current activity in nearby candidate zones. 
# It produces transparent neighbourhood exposure, active-neighbour count and active-neighbour fraction.

zone_lookup = (
    pd.read_parquet(ZONE_LOOKUP_PATH)
    .sort_values("zone_index")
    .reset_index(drop=True)
)

candidate_graph = pd.read_parquet(
    CANDIDATE_GRAPH_PATH
)

zone_order = np.sort(
    operational_panel["zone_index"].unique()
)

time_order = np.sort(
    operational_panel["time_window"].unique()
)

N_ZONES = len(zone_order)
N_HOURS = len(time_order)

assert len(operational_panel) == N_ZONES * N_HOURS

assert np.array_equal(
    zone_order,
    zone_lookup["zone_index"].to_numpy()
)

zone_to_position = {
    zone: position
    for position, zone in enumerate(zone_order)
}

source_positions = (
    candidate_graph["source_zone"]
    .map(zone_to_position)
    .to_numpy(dtype=np.int32)
)

target_positions = (
    candidate_graph["target_zone"]
    .map(zone_to_position)
    .to_numpy(dtype=np.int32)
)

distance_weights = (
    1 / candidate_graph["distance_m"]
).to_numpy(dtype=np.float32)

incoming_weight_total = np.bincount(
    target_positions,
    weights=distance_weights,
    minlength=N_ZONES
)

normalized_weights = (
    distance_weights
    / incoming_weight_total[target_positions]
)

distance_weight_matrix = csr_matrix(
    (
        normalized_weights,
        (source_positions, target_positions)
    ),
    shape=(N_ZONES, N_ZONES)
)

adjacency_matrix = csr_matrix(
    (
        np.ones(
            len(candidate_graph),
            dtype=np.float32
        ),
        (source_positions, target_positions)
    ),
    shape=(N_ZONES, N_ZONES)
)

incident_matrix = (
    operational_panel["incident_count"]
    .to_numpy(dtype=np.float32)
    .reshape(N_HOURS, N_ZONES)
)

active_matrix = (
    incident_matrix > 0
).astype(np.float32)

neighbour_log_exposure = np.asarray(
    distance_weight_matrix.T.dot(
        np.log1p(incident_matrix).T
    ).T
)

active_neighbour_count = np.asarray(
    adjacency_matrix.T.dot(
        active_matrix.T
    ).T
)

incoming_degree = np.asarray(
    adjacency_matrix.sum(axis=0)
).ravel()

active_neighbour_fraction = np.divide(
    active_neighbour_count,
    incoming_degree[None, :],
    out=np.zeros_like(active_neighbour_count),
    where=incoming_degree[None, :] > 0
)

operational_panel[
    "distance_neighbour_exposure"
] = np.expm1(
    neighbour_log_exposure
).ravel().astype("float32")

operational_panel[
    "active_neighbour_count"
] = active_neighbour_count.ravel().astype(
    "int16"
)

operational_panel[
    "active_neighbour_fraction"
] = active_neighbour_fraction.ravel().astype(
    "float32"
)

print(
    operational_panel[
        [
            "distance_neighbour_exposure",
            "active_neighbour_count",
            "active_neighbour_fraction"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
#)

SyntaxError: incomplete input (4031415477.py, line 146)

In [ ]:
# Create transparent disruption evidence

# No severity weights are assigned. We measure the strongest road-interface evidence,
# the number of distinct evidence types present and separate vehicle/offence indicators.
share_columns = [
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share"
]

missing_share_columns = [
    col for col in share_columns
    if col not in operational_panel.columns
]

if missing_share_columns:
    raise KeyError(
        f"Missing context shares: {missing_share_columns}"
    )

road_interface_columns = [
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share"
]

operational_panel[
    "road_interface_evidence"
] = operational_panel[
    road_interface_columns
].max(axis=1).astype("float32")

operational_panel[
    "context_evidence_breadth"
] = operational_panel[
    share_columns
].gt(0).sum(axis=1).astype("int8")

operational_panel[
    "explicit_road_obstruction"
] = operational_panel[
    road_interface_columns
].gt(0).any(axis=1).astype("int8")

operational_panel[
    "large_vehicle_evidence"
] = operational_panel[
    "large_vehicle_share"
].gt(0).astype("int8")

operational_panel[
    "multi_offence_evidence"
] = operational_panel[
    "multi_offence_share"
].gt(0).astype("int8")

context_summary = (
    operational_panel.loc[
        operational_panel["incident_count"] > 0,
        [
            "road_interface_evidence",
            "context_evidence_breadth",
            "large_vehicle_share",
            "multi_offence_share",
            "distance_neighbour_exposure",
            "active_neighbour_count",
            "observation_confidence"
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
    .T
)

evidence_presence_summary = pd.DataFrame({
    "indicator": [
        "explicit_road_obstruction",
        "large_vehicle_evidence",
        "multi_offence_evidence",
        "active_neighbour_present"
    ],
    "active_zone_hour_percent": [
        100 * operational_panel.loc[
            operational_panel["incident_count"] > 0,
            "explicit_road_obstruction"
        ].mean(),
        100 * operational_panel.loc[
            operational_panel["incident_count"] > 0,
            "large_vehicle_evidence"
        ].mean(),
        100 * operational_panel.loc[
            operational_panel["incident_count"] > 0,
            "multi_offence_evidence"
        ].mean(),
        100 * operational_panel.loc[
            operational_panel["incident_count"] > 0,
            "active_neighbour_count"
        ].gt(0).mean()
    ]
})

display(context_summary)
display(evidence_presence_summary)

operational_panel.to_parquet(
    OUTPUT_DIR
    / "operational_risk_components.parquet",
    index=False
)

context_summary.to_csv(
    OUTPUT_DIR
    / "operational_context_summary.csv"
)

evidence_presence_summary.to_csv(
    OUTPUT_DIR
    / "evidence_presence_summary.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR
    / "operational_risk_components.parquet"
)

## Disruption and Confidence Tiers

In [ ]:
# Create transparent disruption-evidence tiers

# The tier is based on logical evidence combinations rather than arbitrary numerical weights. 
# The highest tier requires road-interface evidence together with either a large vehicle or multiple offences

high_impact_context = (
    operational_panel["explicit_road_obstruction"].eq(1)
    & (
        operational_panel["large_vehicle_evidence"].eq(1)
        | operational_panel["multi_offence_evidence"].eq(1)
    )
)

multiple_context = (
    operational_panel["context_evidence_breadth"] >= 2
)

single_context = (
    operational_panel["context_evidence_breadth"] == 1
)

operational_panel["disruption_tier_code"] = np.select(
    [
        high_impact_context,
        multiple_context,
        single_context
    ],
    [3, 2, 1],
    default=0
).astype("int8")

disruption_labels = {
    0: "no_explicit_context",
    1: "single_context_signal",
    2: "multiple_context_signals",
    3: "road_interface_plus_compounding_evidence"
}

operational_panel["disruption_tier"] = (
    operational_panel["disruption_tier_code"]
    .map(disruption_labels)
    .astype("category")
)

display(
    operational_panel.loc[
        operational_panel["incident_count"] > 0,
        "disruption_tier"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .rename("active_zone_hour_percent")
    .to_frame()
)

In [ ]:
# validate the proxy descriptive
context_outcome_summary = (
    operational_panel.loc[
        operational_panel["incident_count"] > 0
    ]
    .groupby(
        ["split_code", "disruption_tier"],
        observed=True
    )
    .agg(
        zone_hours=("zone_index", "size"),
        mean_current_incidents=(
            "incident_count",
            "mean"
        ),
        mean_predicted_pressure=(
            "calibrated_pressure",
            "mean"
        ),
        mean_actual_next_pressure=(
            "actual_pressure",
            "mean"
        ),
        next_hour_active_rate=(
            "actual_pressure",
            lambda x: (x > 0).mean()
        ),
        next_hour_hotspot_rate=(
            "actual_pressure",
            lambda x: (x >= 8).mean()
        ),
        mean_neighbour_exposure=(
            "distance_neighbour_exposure",
            "mean"
        ),
        mean_observation_confidence=(
            "observation_confidence",
            "mean"
        )
    )
    .reset_index()
)

context_outcome_summary["split"] = (
    context_outcome_summary["split_code"]
    .map({
        2: "validation",
        3: "test"
    })
)

display(
    context_outcome_summary[
        [
            "split",
            "disruption_tier",
            "zone_hours",
            "mean_current_incidents",
            "mean_predicted_pressure",
            "mean_actual_next_pressure",
            "next_hour_active_rate",
            "next_hour_hotspot_rate",
            "mean_neighbour_exposure",
            "mean_observation_confidence"
        ]
    ]
)

In [ ]:
# Recreate the validated distance-adjusted ranking

# The first half of validation estimates a nonnegative spatial correction. 
# The second validation half and complete test period remain untouched for evaluation
from sklearn.linear_model import LinearRegression

validation_times = np.sort(
    operational_panel.loc[
        operational_panel["split_code"] == 2,
        "target_time"
    ].unique()
)

validation_midpoint = len(validation_times) // 2

spatial_fit_times = validation_times[
    :validation_midpoint
]

spatial_fit_mask = (
    operational_panel["split_code"].eq(2)
    & operational_panel["target_time"].isin(
        spatial_fit_times
    )
)

X_spatial_fit = np.column_stack([
    np.log1p(
        operational_panel.loc[
            spatial_fit_mask,
            "predicted_pressure"
        ]
    ),
    operational_panel.loc[
        spatial_fit_mask,
        "distance_neighbour_exposure"
    ]
])

y_spatial_fit = np.log1p(
    operational_panel.loc[
        spatial_fit_mask,
        "actual_pressure"
    ]
)

spatial_fit_weights = (
    1
    + np.sqrt(
        operational_panel.loc[
            spatial_fit_mask,
            "actual_pressure"
        ]
    )
)

spatial_combiner = LinearRegression(
    positive=True
)

spatial_combiner.fit(
    X_spatial_fit,
    y_spatial_fit,
    sample_weight=spatial_fit_weights
)

all_spatial_features = np.column_stack([
    np.log1p(
        operational_panel["predicted_pressure"]
    ),
    operational_panel[
        "distance_neighbour_exposure"
    ]
])

spatial_log_prediction = spatial_combiner.predict(
    all_spatial_features
)

operational_panel["spatial_priority_score"] = (
    np.expm1(
        np.maximum(
            spatial_log_prediction,
            0
        )
    )
    .astype("float32")
)

spatial_combiner_summary = pd.DataFrame([{
    "intercept": spatial_combiner.intercept_,
    "local_prediction_coefficient":
        spatial_combiner.coef_[0],
    "distance_exposure_coefficient":
        spatial_combiner.coef_[1],
    "fit_hours": len(spatial_fit_times)
}])

display(spatial_combiner_summary)

In [ ]:
# Confirm the spatial ranking on unseen periods

# This compares raw local rankings with distance-adjusted rankings at K=25

def evaluate_priority_at_25(data, prediction_column):
    ordered = (
        data.sort_values(
            ["target_time", prediction_column],
            ascending=[True, False]
        )
        .copy()
    )

    ordered["hour_rank"] = (
        ordered.groupby("target_time")
        .cumcount()
        .add(1)
    )

    selected = ordered.loc[
        ordered["hour_rank"] <= 25
    ]

    total_pressure = ordered[
        "actual_pressure"
    ].sum()

    total_hotspots = (
        ordered["actual_pressure"] >= 8
    ).sum()

    selected_hotspots = (
        selected["actual_pressure"] >= 8
    ).sum()

    hotspot_hours = (
        ordered.groupby("target_time")[
            "actual_pressure"
        ]
        .max()
        .ge(8)
    )

    selected_hotspot_hours = (
        selected.groupby("target_time")[
            "actual_pressure"
        ]
        .max()
        .ge(8)
        .reindex(
            hotspot_hours.index,
            fill_value=False
        )
    )

    return {
        "pressure_capture_at_25": (
            selected["actual_pressure"].sum()
            / total_pressure
        ),
        "active_zone_recall_at_25": (
            selected["actual_pressure"].gt(0).sum()
            / ordered["actual_pressure"].gt(0).sum()
        ),
        "hotspot_recall_at_25": (
            selected_hotspots / total_hotspots
            if total_hotspots > 0 else 0
        ),
        "hotspot_hit_rate_at_25": (
            selected_hotspot_hours[
                hotspot_hours
            ].mean()
            if hotspot_hours.any() else 0
        )
    }


spatial_validation_rows = []

unseen_validation_times = validation_times[
    validation_midpoint:
]

evaluation_sets = {
    "validation_holdout": (
        operational_panel["split_code"].eq(2)
        & operational_panel["target_time"].isin(
            unseen_validation_times
        )
    ),
    "test": operational_panel["split_code"].eq(3)
}

for period_name, period_mask in evaluation_sets.items():
    period_data = operational_panel.loc[
        period_mask
    ]

    for variant, column in {
        "local_only": "predicted_pressure",
        "local_plus_distance":
            "spatial_priority_score"
    }.items():
        row = evaluate_priority_at_25(
            period_data,
            column
        )

        row["period"] = period_name
        row["variant"] = variant

        spatial_validation_rows.append(row)

spatial_priority_validation = pd.DataFrame(
    spatial_validation_rows
)

display(
    spatial_priority_validation[
        [
            "period",
            "variant",
            "pressure_capture_at_25",
            "active_zone_recall_at_25",
            "hotspot_recall_at_25",
            "hotspot_hit_rate_at_25"
        ]
    ]
)

In [ ]:
# Create confidence-aware action tiers

# Confidence thresholds come only from active validation observations. 
# Ranking is hourly, while disruption evidence and confidence determine the recommended response

confidence_reference = operational_panel.loc[
    operational_panel["split_code"].eq(2)
    & operational_panel["incident_count"].gt(0),
    "observation_confidence"
]

LOW_CONFIDENCE_THRESHOLD = float(
    confidence_reference.quantile(0.25)
)

HIGH_CONFIDENCE_THRESHOLD = float(
    confidence_reference.quantile(0.75)
)

operational_panel["confidence_tier"] = np.select(
    [
        operational_panel[
            "observation_confidence"
        ] >= HIGH_CONFIDENCE_THRESHOLD,

        operational_panel[
            "observation_confidence"
        ] < LOW_CONFIDENCE_THRESHOLD
    ],
    [
        "high_confidence",
        "low_confidence"
    ],
    default="medium_confidence"
)

operational_panel["priority_rank"] = (
    operational_panel.groupby(
        "target_time"
    )["spatial_priority_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype("int16")
)

operational_panel["recommended_response"] = np.select(
    [
        (
            operational_panel["priority_rank"] <= 25
        )
        & operational_panel[
            "confidence_tier"
        ].eq("high_confidence"),

        (
            operational_panel["priority_rank"] <= 25
        )
        & operational_panel[
            "confidence_tier"
        ].eq("medium_confidence"),

        (
            operational_panel["priority_rank"] <= 25
        )
        & operational_panel[
            "confidence_tier"
        ].eq("low_confidence"),

        (
            operational_panel["priority_rank"] <= 50
        )
        & operational_panel[
            "disruption_tier_code"
        ].ge(2),

        operational_panel["priority_rank"] <= 100
    ],
    [
        "immediate_enforcement",
        "priority_patrol",
        "verify_before_enforcement",
        "context_driven_patrol",
        "monitor"
    ],
    default="no_immediate_action"
)

action_summary = (
    operational_panel.groupby(
        "recommended_response"
    )
    .agg(
        zone_hours=("zone_index", "size"),
        mean_priority_rank=(
            "priority_rank",
            "mean"
        ),
        mean_spatial_priority=(
            "spatial_priority_score",
            "mean"
        ),
        mean_hotspot_probability=(
            "hotspot_probability",
            "mean"
        ),
        mean_confidence=(
            "observation_confidence",
            "mean"
        ),
        mean_disruption_tier=(
            "disruption_tier_code",
            "mean"
        ),
        actual_next_pressure=(
            "actual_pressure",
            "mean"
        )
    )
    .reset_index()
    .sort_values("mean_priority_rank")
)

print(
    "Low-confidence threshold:",
    LOW_CONFIDENCE_THRESHOLD
)

print(
    "High-confidence threshold:",
    HIGH_CONFIDENCE_THRESHOLD
)

display(action_summary)

operational_panel.to_parquet(
    OUTPUT_DIR
    / "confidence_aware_operational_table.parquet",
    index=False
)

context_outcome_summary.to_csv(
    OUTPUT_DIR
    / "context_outcome_validation.csv",
    index=False
)

spatial_combiner_summary.to_csv(
    OUTPUT_DIR
    / "spatial_priority_combiner.csv",
    index=False
)

spatial_priority_validation.to_csv(
    OUTPUT_DIR
    / "spatial_priority_validation.csv",
    index=False
)

action_summary.to_csv(
    OUTPUT_DIR
    / "confidence_aware_action_summary.csv",
    index=False
)

print("Phase 6 operational tiers saved.")

## Enforcement Scenarios and Patrol Optimization

In [ ]:
# Consolidate disruption classes and create suppression scenarios
operational_panel["disruption_class_code"] = np.select(
    [
        operational_panel["disruption_tier_code"] >= 2,
        operational_panel["disruption_tier_code"] == 1
    ],
    [2, 1],
    default=0
).astype("int8")

disruption_class_labels = {
    0: "no_explicit_context",
    1: "single_context_signal",
    2: "compounded_disruption_evidence"
}

operational_panel["disruption_class"] = (
    operational_panel["disruption_class_code"]
    .map(disruption_class_labels)
    .astype("category")
)

SUPPRESSION_SCENARIOS = {
    "30pct": 0.30,
    "60pct": 0.60,
    "90pct": 0.90
}

for scenario_name, suppression in (
    SUPPRESSION_SCENARIOS.items()
):
    operational_panel[
        f"local_pressure_removed_{scenario_name}"
    ] = (
        suppression
        * operational_panel["calibrated_pressure"]
    ).astype("float32")

    operational_panel[
        f"local_pressure_remaining_{scenario_name}"
    ] = (
        (1 - suppression)
        * operational_panel["calibrated_pressure"]
    ).astype("float32")

display(
    operational_panel.loc[
        operational_panel["incident_count"] > 0,
        "disruption_class"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .rename("active_zone_hour_percent")
    .to_frame()
)

In [ ]:
# summarize scenario based local enforcement gain
scenario_rows = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    split_data = operational_panel.loc[
        operational_panel["split_code"] == split_code
    ]

    for scenario_name, suppression in (
        SUPPRESSION_SCENARIOS.items()
    ):
        gain_column = (
            f"local_pressure_removed_{scenario_name}"
        )

        for response, response_data in (
            split_data.groupby(
                "recommended_response"
            )
        ):
            scenario_rows.append({
                "split": split_name,
                "scenario": scenario_name,
                "suppression_fraction": suppression,
                "recommended_response": response,
                "zone_hours": len(response_data),
                "mean_local_pressure_removed":
                    response_data[gain_column].mean(),
                "total_local_pressure_removed":
                    response_data[gain_column].sum(),
                "mean_disruption_class":
                    response_data[
                        "disruption_class_code"
                    ].mean(),
                "mean_observation_confidence":
                    response_data[
                        "observation_confidence"
                    ].mean()
            })

scenario_gain_summary = pd.DataFrame(
    scenario_rows
)

display(
    scenario_gain_summary.loc[
        scenario_gain_summary[
            "scenario"
        ] == "60pct"
    ].sort_values(
        [
            "split",
            "mean_local_pressure_removed"
        ],
        ascending=[True, False]
    )
)

In [ ]:
# Build 500 m patrol de-duplication neighbourhoods

# The 500 m radius equals the median validated spatial-edge distance.
# It is used only to prevent multiple patrol recommendations from clustering in adjacent 250 m cells.

DEDUPLICATION_RADIUS_M = 500

deduplication_edges = candidate_graph.loc[
    candidate_graph["distance_m"]
    <= DEDUPLICATION_RADIUS_M
].copy()

dedup_source_positions = (
    deduplication_edges["source_zone"]
    .map(zone_to_position)
    .to_numpy(dtype=np.int32)
)

dedup_target_positions = (
    deduplication_edges["target_zone"]
    .map(zone_to_position)
    .to_numpy(dtype=np.int32)
)

coverage_matrix = csr_matrix(
    (
        np.ones(
            len(deduplication_edges),
            dtype=np.float32
        ),
        (
            dedup_source_positions,
            dedup_target_positions
        )
    ),
    shape=(N_ZONES, N_ZONES)
)

# Every patrol location covers its own zone.
coverage_matrix = (
    coverage_matrix
    + csr_matrix(
        (
            np.ones(N_ZONES, dtype=np.float32),
            (
                np.arange(N_ZONES),
                np.arange(N_ZONES)
            )
        ),
        shape=(N_ZONES, N_ZONES)
    )
)

coverage_matrix.data[:] = 1
coverage_matrix.eliminate_zeros()

coverage_sizes = np.asarray(
    coverage_matrix.sum(axis=1)
).ravel()

print("De-duplication radius:", DEDUPLICATION_RADIUS_M)
print(
    pd.Series(coverage_sizes).describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
# Generate naive and redundancy-aware patrol selections

# The greedy strategy selects zones that cover the greatest amount of currently uncovered predicted priority. 
# It considers only the top 100 forecast candidates each hour and produces prefixes for K = 5, 10 and 25

PATROL_K_VALUES = [5, 10, 25]
MAX_PATROLS = max(PATROL_K_VALUES)
CANDIDATE_POOL_SIZE = 100

priority_matrix = (
    operational_panel["spatial_priority_score"]
    .to_numpy(dtype=np.float32)
    .reshape(N_HOURS, N_ZONES)
)

gain_60_matrix = (
    operational_panel[
        "local_pressure_removed_60pct"
    ]
    .to_numpy(dtype=np.float32)
    .reshape(N_HOURS, N_ZONES)
)

confidence_matrix = (
    operational_panel["observation_confidence"]
    .to_numpy(dtype=np.float32)
    .reshape(N_HOURS, N_ZONES)
)

disruption_matrix = (
    operational_panel["disruption_class_code"]
    .to_numpy(dtype=np.int8)
    .reshape(N_HOURS, N_ZONES)
)

target_time_order = np.sort(
    operational_panel["target_time"].unique()
)

split_by_hour = (
    operational_panel["split_code"]
    .to_numpy(dtype=np.int8)
    .reshape(N_HOURS, N_ZONES)[:, 0]
)

selection_rows = []

for hour_position in range(N_HOURS):
    priority = priority_matrix[hour_position]

    candidate_pool = np.argpartition(
        -priority,
        kth=min(
            CANDIDATE_POOL_SIZE,
            N_ZONES
        ) - 1
    )[:CANDIDATE_POOL_SIZE]

    candidate_pool = candidate_pool[
        np.argsort(
            -priority[candidate_pool]
        )
    ]

    naive_order = candidate_pool[
        :MAX_PATROLS
    ]

    uncovered_priority = priority.copy()
    greedy_order = []

    available_candidates = set(
        candidate_pool.tolist()
    )

    for selection_number in range(
        MAX_PATROLS
    ):
        marginal_coverage = np.asarray(
            coverage_matrix.dot(
                uncovered_priority
            )
        ).ravel()

        available_array = np.fromiter(
            available_candidates,
            dtype=np.int32
        )

        if len(available_array) == 0:
            break

        candidate_scores = (
            marginal_coverage[available_array]
            + 1e-6
            * priority[available_array]
        )

        best_position = available_array[
            np.argmax(candidate_scores)
        ]

        greedy_order.append(best_position)
        available_candidates.remove(
            int(best_position)
        )

        covered_positions = (
            coverage_matrix[
                best_position
            ].indices
        )

        uncovered_priority[
            covered_positions
        ] = 0

    for strategy, selected_order in [
        ("naive_top_priority", naive_order),
        (
            "redundancy_aware_coverage",
            np.asarray(
                greedy_order,
                dtype=np.int32
            )
        )
    ]:
        for order, zone_position in enumerate(
            selected_order,
            start=1
        ):
            selection_rows.append({
                "target_time":
                    target_time_order[hour_position],
                "split_code":
                    split_by_hour[hour_position],
                "strategy": strategy,
                "selection_order": order,
                "zone_index":
                    zone_order[zone_position],
                "zone_position": zone_position,
                "spatial_priority_score":
                    priority[zone_position],
                "scenario_local_gain_60pct":
                    gain_60_matrix[
                        hour_position,
                        zone_position
                    ],
                "observation_confidence":
                    confidence_matrix[
                        hour_position,
                        zone_position
                    ],
                "disruption_class_code":
                    disruption_matrix[
                        hour_position,
                        zone_position
                    ],
                "marginal_predicted_coverage":
                    float(
                        np.asarray(
                            coverage_matrix[
                                zone_position
                            ].dot(
                                priority
                            )
                        ).ravel()[0]
                    )
            })

patrol_selections = pd.DataFrame(
    selection_rows
)

print("Patrol selection rows:", len(patrol_selections))

display(
    patrol_selections.loc[
        patrol_selections[
            "target_time"
        ] == patrol_selections[
            "target_time"
        ].max()
    ].head(20)
)

In [ ]:
# compare naive and redundancy aware recommendation
actual_matrix = (
    operational_panel["actual_pressure"]
    .to_numpy(dtype=np.float32)
    .reshape(N_HOURS, N_ZONES)
)

coordinate_matrix = zone_lookup[
    ["centroid_x_m", "centroid_y_m"]
].to_numpy(dtype=np.float64)

evaluation_rows = []

period_masks = {
    "validation_holdout": (
        (split_by_hour == 2)
        & np.isin(
            target_time_order,
            unseen_validation_times
        )
    ),
    "test": split_by_hour == 3
}

for period_name, period_hour_mask in (
    period_masks.items()
):
    period_hour_positions = np.flatnonzero(
        period_hour_mask
    )

    for strategy in [
        "naive_top_priority",
        "redundancy_aware_coverage"
    ]:
        strategy_data = patrol_selections.loc[
            patrol_selections["strategy"]
            == strategy
        ]

        for k in PATROL_K_VALUES:
            direct_pressure = 0.0
            area_pressure = 0.0
            total_pressure = 0.0

            direct_hotspots = 0
            area_hotspots = 0
            total_hotspots = 0

            unique_zones_covered = []
            overlap_rates = []
            pairwise_distances = []

            for hour_position in (
                period_hour_positions
            ):
                target_time = (
                    target_time_order[
                        hour_position
                    ]
                )

                selected_positions = (
                    strategy_data.loc[
                        (
                            strategy_data[
                                "target_time"
                            ] == target_time
                        )
                        & (
                            strategy_data[
                                "selection_order"
                            ] <= k
                        ),
                        "zone_position"
                    ]
                    .to_numpy(dtype=np.int32)
                )

                actual = actual_matrix[
                    hour_position
                ]

                covered = np.zeros(
                    N_ZONES,
                    dtype=bool
                )

                individual_coverage_total = 0

                for selected_position in (
                    selected_positions
                ):
                    covered_positions = (
                        coverage_matrix[
                            selected_position
                        ].indices
                    )

                    covered[
                        covered_positions
                    ] = True

                    individual_coverage_total += (
                        len(covered_positions)
                    )

                direct_pressure += actual[
                    selected_positions
                ].sum()

                area_pressure += actual[
                    covered
                ].sum()

                total_pressure += actual.sum()

                hotspot_mask = actual >= 8

                direct_hotspots += hotspot_mask[
                    selected_positions
                ].sum()

                area_hotspots += hotspot_mask[
                    covered
                ].sum()

                total_hotspots += hotspot_mask.sum()

                unique_count = int(
                    covered.sum()
                )

                unique_zones_covered.append(
                    unique_count
                )

                overlap_rates.append(
                    1
                    - unique_count
                    / individual_coverage_total
                    if individual_coverage_total > 0
                    else 0
                )

                if len(selected_positions) > 1:
                    selected_coordinates = (
                        coordinate_matrix[
                            selected_positions
                        ]
                    )

                    differences = (
                        selected_coordinates[:, None, :]
                        - selected_coordinates[
                            None, :, :
                        ]
                    )

                    distance_matrix = np.sqrt(
                        np.sum(
                            differences ** 2,
                            axis=2
                        )
                    )

                    upper_triangle = (
                        np.triu_indices(
                            len(selected_positions),
                            k=1
                        )
                    )

                    pairwise_distances.append(
                        distance_matrix[
                            upper_triangle
                        ].mean()
                    )

            evaluation_rows.append({
                "period": period_name,
                "strategy": strategy,
                "k": k,
                "direct_pressure_capture": (
                    direct_pressure
                    / total_pressure
                ),
                "area_pressure_coverage": (
                    area_pressure
                    / total_pressure
                ),
                "direct_hotspot_recall": (
                    direct_hotspots
                    / total_hotspots
                    if total_hotspots > 0
                    else 0
                ),
                "area_hotspot_coverage": (
                    area_hotspots
                    / total_hotspots
                    if total_hotspots > 0
                    else 0
                ),
                "mean_unique_zones_covered":
                    np.mean(
                        unique_zones_covered
                    ),
                "mean_overlap_rate":
                    np.mean(overlap_rates),
                "mean_pairwise_distance_m":
                    np.mean(
                        pairwise_distances
                    )
            })

patrol_strategy_evaluation = pd.DataFrame(
    evaluation_rows
)

display(
    patrol_strategy_evaluation.sort_values(
        ["period", "k", "strategy"]
    )
)

scenario_gain_summary.to_csv(
    OUTPUT_DIR
    / "local_enforcement_scenario_summary.csv",
    index=False
)

patrol_selections.to_parquet(
    OUTPUT_DIR
    / "patrol_location_selections.parquet",
    index=False
)

patrol_strategy_evaluation.to_csv(
    OUTPUT_DIR
    / "patrol_strategy_evaluation.csv",
    index=False
)

operational_panel.to_parquet(
    OUTPUT_DIR
    / "operational_table_with_scenarios.parquet",
    index=False
)

print("Scenario and patrol outputs saved.")

## Balanced Patrol Optimization

The optimizer balances:

direct pressure at the selected zone;
additional predicted pressure covered within 500 m.

In [ ]:
# define the balance greedy selector
BALANCE_ALPHA_GRID = [
    0.00,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00
]

PATROL_K_VALUES = [5, 10, 25]
MAX_PATROLS = max(PATROL_K_VALUES)
CANDIDATE_POOL_SIZE = 100


def balanced_patrol_order(
    priority,
    alpha,
    candidate_pool_size=100,
    maximum_patrols=25
):
    pool_size = min(
        candidate_pool_size,
        len(priority)
    )

    candidate_pool = np.argpartition(
        -priority,
        kth=pool_size - 1
    )[:pool_size]

    candidate_pool = candidate_pool[
        np.argsort(-priority[candidate_pool])
    ]

    available = set(candidate_pool.tolist())
    uncovered_priority = priority.copy()

    selected_positions = []
    selection_details = []

    for selection_order in range(
        1,
        maximum_patrols + 1
    ):
        if not available:
            break

        available_array = np.fromiter(
            available,
            dtype=np.int32
        )

        direct_scores = priority[
            available_array
        ]

        marginal_coverage_scores = np.asarray(
            coverage_matrix[
                available_array
            ].dot(uncovered_priority)
        ).ravel()

        direct_max = direct_scores.max()
        coverage_max = (
            marginal_coverage_scores.max()
        )

        direct_normalized = np.divide(
            direct_scores,
            direct_max,
            out=np.zeros_like(
                direct_scores,
                dtype=np.float32
            ),
            where=direct_max > 0
        )

        coverage_normalized = np.divide(
            marginal_coverage_scores,
            coverage_max,
            out=np.zeros_like(
                marginal_coverage_scores,
                dtype=np.float32
            ),
            where=coverage_max > 0
        )

        combined_utility = (
            alpha * direct_normalized
            + (1 - alpha)
            * coverage_normalized
            + 1e-8 * direct_normalized
        )

        best_local_index = int(
            np.argmax(combined_utility)
        )

        best_position = int(
            available_array[
                best_local_index
            ]
        )

        selected_positions.append(
            best_position
        )

        selection_details.append({
            "selection_order":
                selection_order,
            "zone_position":
                best_position,
            "direct_priority":
                float(
                    priority[best_position]
                ),
            "marginal_predicted_coverage":
                float(
                    marginal_coverage_scores[
                        best_local_index
                    ]
                ),
            "combined_utility":
                float(
                    combined_utility[
                        best_local_index
                    ]
                )
        })

        available.remove(best_position)

        covered_positions = (
            coverage_matrix[
                best_position
            ].indices
        )

        uncovered_priority[
            covered_positions
        ] = 0

    return selected_positions, selection_details

In [ ]:
# Generate selections for each balance value

# slections are generated for all validation and test hours. The test period is not used to choose alpha
balanced_selection_rows = []

for hour_position in range(N_HOURS):
    priority = priority_matrix[
        hour_position
    ]

    for alpha in BALANCE_ALPHA_GRID:
        selected_positions, details = (
            balanced_patrol_order(
                priority=priority,
                alpha=alpha,
                candidate_pool_size=(
                    CANDIDATE_POOL_SIZE
                ),
                maximum_patrols=MAX_PATROLS
            )
        )

        for detail in details:
            zone_position = detail[
                "zone_position"
            ]

            balanced_selection_rows.append({
                "target_time":
                    target_time_order[
                        hour_position
                    ],
                "split_code":
                    split_by_hour[
                        hour_position
                    ],
                "alpha": alpha,
                "selection_order":
                    detail[
                        "selection_order"
                    ],
                "zone_index":
                    zone_order[
                        zone_position
                    ],
                "zone_position":
                    zone_position,
                "spatial_priority_score":
                    priority[
                        zone_position
                    ],
                "scenario_local_gain_60pct":
                    gain_60_matrix[
                        hour_position,
                        zone_position
                    ],
                "observation_confidence":
                    confidence_matrix[
                        hour_position,
                        zone_position
                    ],
                "disruption_class_code":
                    disruption_matrix[
                        hour_position,
                        zone_position
                    ],
                "direct_priority":
                    detail[
                        "direct_priority"
                    ],
                "marginal_predicted_coverage":
                    detail[
                        "marginal_predicted_coverage"
                    ],
                "combined_utility":
                    detail[
                        "combined_utility"
                    ]
            })

balanced_patrol_selections = pd.DataFrame(
    balanced_selection_rows
)

print(
    "Balanced selection rows:",
    len(balanced_patrol_selections)
)

display(
    balanced_patrol_selections.head()
)

In [ ]:
# evaluate each alpha on validation
selection_lookup = {
    key: group.sort_values(
        "selection_order"
    )["zone_position"].to_numpy(
        dtype=np.int32
    )
    for key, group in (
        balanced_patrol_selections.groupby(
            ["alpha", "target_time"]
        )
    )
}


def evaluate_balanced_strategy(
    alpha,
    hour_positions,
    k
):
    direct_pressure = 0.0
    area_pressure = 0.0
    total_pressure = 0.0

    direct_hotspots = 0
    area_hotspots = 0
    total_hotspots = 0

    overlap_rates = []
    unique_coverage_counts = []

    for hour_position in hour_positions:
        target_time = target_time_order[
            hour_position
        ]

        selected_positions = (
            selection_lookup[
                (alpha, target_time)
            ][:k]
        )

        actual = actual_matrix[
            hour_position
        ]

        covered = np.zeros(
            N_ZONES,
            dtype=bool
        )

        individual_coverage_total = 0

        for selected_position in (
            selected_positions
        ):
            covered_positions = (
                coverage_matrix[
                    selected_position
                ].indices
            )

            covered[
                covered_positions
            ] = True

            individual_coverage_total += len(
                covered_positions
            )

        direct_pressure += actual[
            selected_positions
        ].sum()

        area_pressure += actual[
            covered
        ].sum()

        total_pressure += actual.sum()

        hotspot_mask = actual >= 8

        direct_hotspots += hotspot_mask[
            selected_positions
        ].sum()

        area_hotspots += hotspot_mask[
            covered
        ].sum()

        total_hotspots += hotspot_mask.sum()

        unique_count = int(
            covered.sum()
        )

        unique_coverage_counts.append(
            unique_count
        )

        overlap_rates.append(
            1
            - unique_count
            / individual_coverage_total
            if individual_coverage_total > 0
            else 0
        )

    direct_pressure_capture = (
        direct_pressure / total_pressure
    )

    area_pressure_coverage = (
        area_pressure / total_pressure
    )

    direct_hotspot_recall = (
        direct_hotspots / total_hotspots
        if total_hotspots > 0 else 0
    )

    area_hotspot_coverage = (
        area_hotspots / total_hotspots
        if total_hotspots > 0 else 0
    )

    balanced_score = (
        max(direct_pressure_capture, 1e-12)
        * max(area_pressure_coverage, 1e-12)
        * max(direct_hotspot_recall, 1e-12)
        * max(area_hotspot_coverage, 1e-12)
    ) ** 0.25

    return {
        "alpha": alpha,
        "k": k,
        "direct_pressure_capture":
            direct_pressure_capture,
        "area_pressure_coverage":
            area_pressure_coverage,
        "direct_hotspot_recall":
            direct_hotspot_recall,
        "area_hotspot_coverage":
            area_hotspot_coverage,
        "mean_unique_zones_covered":
            np.mean(
                unique_coverage_counts
            ),
        "mean_overlap_rate":
            np.mean(overlap_rates),
        "balanced_score":
            balanced_score
    }


validation_hour_positions = np.flatnonzero(
    (
        split_by_hour == 2
    )
    & np.isin(
        target_time_order,
        unseen_validation_times
    )
)

validation_balance_rows = []

for alpha in BALANCE_ALPHA_GRID:
    for k in PATROL_K_VALUES:
        validation_balance_rows.append(
            evaluate_balanced_strategy(
                alpha=alpha,
                hour_positions=(
                    validation_hour_positions
                ),
                k=k
            )
        )

validation_balance_results = pd.DataFrame(
    validation_balance_rows
)

alpha_selection_summary = (
    validation_balance_results.groupby(
        "alpha"
    )
    .agg(
        mean_balanced_score=(
            "balanced_score",
            "mean"
        ),
        mean_direct_pressure_capture=(
            "direct_pressure_capture",
            "mean"
        ),
        mean_area_pressure_coverage=(
            "area_pressure_coverage",
            "mean"
        ),
        mean_direct_hotspot_recall=(
            "direct_hotspot_recall",
            "mean"
        ),
        mean_area_hotspot_coverage=(
            "area_hotspot_coverage",
            "mean"
        ),
        mean_overlap_rate=(
            "mean_overlap_rate",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        [
            "mean_balanced_score",
            "alpha"
        ],
        ascending=[False, False]
    )
)

BEST_BALANCE_ALPHA = float(
    alpha_selection_summary.iloc[0][
        "alpha"
    ]
)

display(validation_balance_results)
display(alpha_selection_summary)

print(
    "Validation-selected alpha:",
    BEST_BALANCE_ALPHA
)

In [ ]:
# Evaluate the frozen balanced strategy on test

# This compares the selected compromise with the two extremes. alpha=1 prioritizes direct pressure, 
# while alpha=0 prioritizes spatial coverage
test_hour_positions = np.flatnonzero(
    split_by_hour == 3
)

final_comparison_alphas = sorted(
    set([
        0.0,
        BEST_BALANCE_ALPHA,
        1.0
    ])
)

final_balance_rows = []

for period_name, hour_positions in [
    (
        "validation_holdout",
        validation_hour_positions
    ),
    ("test", test_hour_positions)
]:
    for alpha in final_comparison_alphas:
        for k in PATROL_K_VALUES:
            row = evaluate_balanced_strategy(
                alpha=alpha,
                hour_positions=hour_positions,
                k=k
            )

            row["period"] = period_name

            if alpha == 0:
                row["strategy"] = (
                    "coverage_extreme"
                )
            elif alpha == 1:
                row["strategy"] = (
                    "direct_risk_extreme"
                )
            else:
                row["strategy"] = (
                    "validation_selected_balance"
                )

            final_balance_rows.append(row)

balanced_patrol_evaluation = pd.DataFrame(
    final_balance_rows
)

display(
    balanced_patrol_evaluation[
        [
            "period",
            "strategy",
            "alpha",
            "k",
            "direct_pressure_capture",
            "area_pressure_coverage",
            "direct_hotspot_recall",
            "area_hotspot_coverage",
            "mean_unique_zones_covered",
            "mean_overlap_rate",
            "balanced_score"
        ]
    ].sort_values(
        ["period", "k", "alpha"]
    )
)

final_balanced_selections = (
    balanced_patrol_selections.loc[
        balanced_patrol_selections[
            "alpha"
        ] == BEST_BALANCE_ALPHA
    ]
    .copy()
)

validation_balance_results.to_csv(
    OUTPUT_DIR
    / "patrol_balance_validation_grid.csv",
    index=False
)

alpha_selection_summary.to_csv(
    OUTPUT_DIR
    / "patrol_balance_alpha_selection.csv",
    index=False
)

balanced_patrol_evaluation.to_csv(
    OUTPUT_DIR
    / "balanced_patrol_evaluation.csv",
    index=False
)

final_balanced_selections.to_parquet(
    OUTPUT_DIR
    / "final_balanced_patrol_selections.parquet",
    index=False
)

print(
    "Final balance alpha:",
    BEST_BALANCE_ALPHA
)

print(
    "Final balanced selections:",
    len(final_balanced_selections)
)

## Final Explainable Enforcement Table

In [ ]:
# Keep only patrol-selection fields that are not already stored
# in the operational panel. This prevents duplicate-column suffixes.
selection_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "alpha",
    "selection_order",
    "zone_position",
    "direct_priority",
    "marginal_predicted_coverage",
    "combined_utility"
]

missing_selection_columns = [
    col for col in selection_columns
    if col not in final_balanced_selections.columns
]

if missing_selection_columns:
    raise KeyError(
        "Missing columns in final_balanced_selections: "
        f"{missing_selection_columns}"
    )

selection_base = final_balanced_selections[
    selection_columns
].copy()


# Load forecasting, context, confidence and scenario information
# from one authoritative source: operational_panel.
recommendation_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "time_window",
    "incident_count",
    "unique_vehicles",
    "predicted_pressure",
    "calibrated_pressure",
    "spatial_priority_score",
    "hotspot_probability",
    "distance_neighbour_exposure",
    "active_neighbour_count",
    "active_neighbour_fraction",
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share",
    "explicit_road_obstruction",
    "large_vehicle_evidence",
    "multi_offence_evidence",
    "context_evidence_breadth",
    "disruption_class_code",
    "disruption_class",
    "observation_confidence",
    "confidence_tier",
    "observer_bias_penalty",
    "duplicate_capture_rate",
    "cold_start_zone",
    "current_burst_z",
    "relative_to_same_hour",
    "short_term_trend",
    "consecutive_active_hours_before",
    "recommended_response",
    "local_pressure_removed_30pct",
    "local_pressure_removed_60pct",
    "local_pressure_removed_90pct",
    "actual_pressure"
]

recommendation_columns = [
    col for col in recommendation_columns
    if col in operational_panel.columns
]

required_operational_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "calibrated_pressure",
    "spatial_priority_score"
]

missing_operational_columns = [
    col for col in required_operational_columns
    if col not in recommendation_columns
]

if missing_operational_columns:
    raise KeyError(
        "Missing required columns in operational_panel: "
        f"{missing_operational_columns}"
    )


# Include split_code as a merge key. This verifies that recommendations
# are matched to evidence from the correct chronological split.
final_recommendations = selection_base.merge(
    operational_panel[recommendation_columns],
    on=[
        "zone_index",
        "target_time",
        "split_code"
    ],
    how="left",
    validate="one_to_one"
)

assert len(final_recommendations) == len(selection_base)

assert final_recommendations[
    "calibrated_pressure"
].notna().all()

assert final_recommendations[
    "spatial_priority_score"
].notna().all()


# Add the zone identifier and coordinates required by the prototype map.
spatial_columns = [
    "zone_index",
    "zone_250m",
    "latitude",
    "longitude",
    "centroid_x_m",
    "centroid_y_m"
]

spatial_columns = [
    col for col in spatial_columns
    if col in zone_lookup.columns
]

if "zone_index" not in spatial_columns:
    raise KeyError(
        "zone_index is missing from zone_lookup."
    )

final_recommendations = final_recommendations.merge(
    zone_lookup[spatial_columns],
    on="zone_index",
    how="left",
    validate="many_to_one"
)


# Add readable split labels.
final_recommendations["split"] = (
    final_recommendations["split_code"]
    .map({
        2: "validation",
        3: "test"
    })
)

assert final_recommendations[
    "split"
].notna().all()


# The same ordered list supports different patrol capacities.
final_recommendations["selected_for_k5"] = (
    final_recommendations[
        "selection_order"
    ] <= 5
)

final_recommendations["selected_for_k10"] = (
    final_recommendations[
        "selection_order"
    ] <= 10
)

final_recommendations["selected_for_k25"] = True


# Final integrity checks.
assert len(final_recommendations) == 27_200

assert not final_recommendations.duplicated(
    ["target_time", "zone_index"]
).any()

assert (
    final_recommendations.groupby(
        "target_time"
    ).size() == 25
).all()

assert final_recommendations[
    "selection_order"
].between(1, 25).all()


print(
    "Final recommendation rows:",
    len(final_recommendations)
)

print(
    "Target hours:",
    final_recommendations[
        "target_time"
    ].nunique()
)

print(
    "Recommendations per hour:",
    final_recommendations.groupby(
        "target_time"
    ).size().unique()
)

print(
    "Shape:",
    final_recommendations.shape
)

display(
    final_recommendations[
        [
            "target_time",
            "selection_order",
            "zone_index",
            "split",
            "calibrated_pressure",
            "spatial_priority_score",
            "observation_confidence",
            "disruption_class"
        ]
    ].head(10)
)

In [ ]:
# Generate transparent evidence explanations

# These explanations use direct, human-readable conditions. They do not claim that any single feature caused the forecast.
def create_evidence_text(row):
    evidence = []

    if row["incident_count"] > 0:
        evidence.append(
            f"{int(row['incident_count'])} current incident(s)"
        )

    if row.get("current_burst_z", 0) >= 2:
        evidence.append(
            "unusual burst above recent baseline"
        )

    if row.get("relative_to_same_hour", 0) >= 2:
        evidence.append(
            "at least twice the usual same-hour activity"
        )

    if row.get(
        "consecutive_active_hours_before",
        0
    ) >= 2:
        evidence.append(
            "persistent activity across recent hours"
        )

    if row.get("active_neighbour_count", 0) >= 2:
        evidence.append(
            f"{int(row['active_neighbour_count'])} nearby active zones"
        )

    if row.get("explicit_road_obstruction", 0) == 1:
        evidence.append(
            "road-interface obstruction evidence"
        )

    if row.get("large_vehicle_evidence", 0) == 1:
        evidence.append(
            "large-vehicle involvement"
        )

    if row.get("multi_offence_evidence", 0) == 1:
        evidence.append(
            "multiple-offence evidence"
        )

    if not evidence:
        evidence.append(
            "forecast-only signal with no current contextual evidence"
        )

    return "; ".join(evidence)


final_recommendations["evidence_summary"] = (
    final_recommendations.apply(
        create_evidence_text,
        axis=1
    )
)

final_recommendations["patrol_action"] = np.select(
    [
        final_recommendations[
            "confidence_tier"
        ].eq("low_confidence"),

        (
            final_recommendations[
                "incident_count"
            ].gt(0)
            & final_recommendations[
                "disruption_class_code"
            ].eq(2)
        ),

        final_recommendations[
            "incident_count"
        ].gt(0)
    ],
    [
        "verify_before_enforcement",
        "targeted_obstruction_enforcement",
        "active_parking_enforcement"
    ],
    default="preventive_patrol"
)

final_recommendations["confidence_note"] = np.select(
    [
        final_recommendations[
            "confidence_tier"
        ].eq("low_confidence"),

        final_recommendations[
            "confidence_tier"
        ].eq("high_confidence")
    ],
    [
        "limited observation reliability; verify on site",
        "strong historical observation support"
    ],
    default="moderate observation support"
)

display(
    final_recommendations[
        [
            "target_time",
            "selection_order",
            "zone_index",
            "spatial_priority_score",
            "calibrated_pressure",
            "hotspot_probability",
            "disruption_class",
            "observation_confidence",
            "patrol_action",
            "evidence_summary"
        ]
    ].head(10)
)

In [ ]:
# Create the final latest-hour enforcement table

# The operational table excludes future ground truth. actual_pressure is kept only in a separate evaluation export.
latest_target_time = final_recommendations[
    "target_time"
].max()

latest_enforcement_table = (
    final_recommendations.loc[
        final_recommendations["target_time"]
        == latest_target_time
    ]
    .sort_values("selection_order")
    .reset_index(drop=True)
)

display_columns = [
    "selection_order",
    "zone_index",
    "zone_250m",
    "latitude",
    "longitude",
    "calibrated_pressure",
    "spatial_priority_score",
    "hotspot_probability",
    "distance_neighbour_exposure",
    "active_neighbour_count",
    "disruption_class",
    "observation_confidence",
    "patrol_action",
    "local_pressure_removed_60pct",
    "evidence_summary"
]

display_columns = [
    col for col in display_columns
    if col in latest_enforcement_table.columns
]

display(
    latest_enforcement_table[
        display_columns
    ]
)

evaluation_columns = [
    "actual_pressure"
]

operational_export = (
    final_recommendations.drop(
        columns=evaluation_columns,
        errors="ignore"
    )
)

operational_export.to_parquet(
    OUTPUT_DIR
    / "final_patrol_recommendations.parquet",
    index=False
)

final_recommendations.to_parquet(
    OUTPUT_DIR
    / "final_patrol_recommendations_with_evaluation.parquet",
    index=False
)

latest_enforcement_table.drop(
    columns=evaluation_columns,
    errors="ignore"
).to_csv(
    OUTPUT_DIR
    / "latest_enforcement_recommendations.csv",
    index=False
)

print("Latest target hour:", latest_target_time)

In [ ]:
# summarize the final operational recommendation

final_operational_rows = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    split_recommendations = (
        final_recommendations.loc[
            final_recommendations["split_code"]
            == split_code
        ]
    )

    for k in [5, 10, 25]:
        selected = split_recommendations.loc[
            split_recommendations[
                "selection_order"
            ] <= k
        ]

        final_operational_rows.append({
            "split": split_name,
            "k": k,
            "selected_zone_hours": len(selected),
            "mean_calibrated_pressure":
                selected[
                    "calibrated_pressure"
                ].mean(),
            "mean_spatial_priority":
                selected[
                    "spatial_priority_score"
                ].mean(),
            "total_scenario_gain_30pct":
                selected[
                    "local_pressure_removed_30pct"
                ].sum(),
            "total_scenario_gain_60pct":
                selected[
                    "local_pressure_removed_60pct"
                ].sum(),
            "total_scenario_gain_90pct":
                selected[
                    "local_pressure_removed_90pct"
                ].sum(),
            "compounded_disruption_percent":
                100
                * selected[
                    "disruption_class_code"
                ].eq(2).mean(),
            "low_confidence_percent":
                100
                * selected[
                    "confidence_tier"
                ].eq("low_confidence").mean(),
            "preventive_patrol_percent":
                100
                * selected[
                    "patrol_action"
                ].eq("preventive_patrol").mean(),
            "mean_observation_confidence":
                selected[
                    "observation_confidence"
                ].mean()
        })

final_operational_summary = pd.DataFrame(
    final_operational_rows
)

display(final_operational_summary)

final_operational_summary.to_csv(
    OUTPUT_DIR
    / "final_operational_summary.csv",
    index=False
)

print("Final explainable enforcement outputs saved.")

## Adaptive Patrol Deployment

In [ ]:
from sklearn.metrics import precision_recall_curve

dispatch_validation_mask = (
    operational_panel["split_code"].eq(2)
    & operational_panel["target_time"].isin(
        unseen_validation_times
    )
)

dispatch_validation = operational_panel.loc[
    dispatch_validation_mask,
    [
        "spatial_priority_score",
        "actual_pressure"
    ]
].copy()

dispatch_actual = (
    dispatch_validation["actual_pressure"] > 0
).astype("int8").to_numpy()

dispatch_scores = dispatch_validation[
    "spatial_priority_score"
].to_numpy(dtype=np.float32)

precision, recall, thresholds = (
    precision_recall_curve(
        dispatch_actual,
        dispatch_scores
    )
)

threshold_precision = precision[:-1]
threshold_recall = recall[:-1]

threshold_f1 = np.divide(
    2 * threshold_precision * threshold_recall,
    threshold_precision + threshold_recall,
    out=np.zeros_like(threshold_precision),
    where=(
        threshold_precision
        + threshold_recall
    ) > 0
)

best_threshold_position = int(
    np.argmax(threshold_f1)
)

MIN_DISPATCH_SCORE = float(
    thresholds[best_threshold_position]
)

dispatch_threshold_summary = pd.DataFrame([{
    "minimum_dispatch_score":
        MIN_DISPATCH_SCORE,
    "validation_precision":
        threshold_precision[
            best_threshold_position
        ],
    "validation_recall":
        threshold_recall[
            best_threshold_position
        ],
    "validation_f1":
        threshold_f1[
            best_threshold_position
        ],
    "validation_active_prevalence":
        dispatch_actual.mean(),
    "selected_validation_rows":
        int(
            (
                dispatch_scores
                >= MIN_DISPATCH_SCORE
            ).sum()
        )
}])

display(dispatch_threshold_summary)

print(
    "Minimum dispatch score:",
    MIN_DISPATCH_SCORE
)

In [ ]:
# Generate adaptive balanced selections

# This reruns the α=0.6 optimizer using only locations above the validation-selected threshold. 
# Quiet hours may therefore use fewer than 5, 10 or 25 patrols—or none

def adaptive_balanced_patrol_order(
    priority,
    alpha,
    minimum_score,
    candidate_pool_size=100,
    maximum_patrols=25
):
    eligible_positions = np.flatnonzero(
        priority >= minimum_score
    )

    if len(eligible_positions) == 0:
        return []

    pool_size = min(
        candidate_pool_size,
        len(eligible_positions)
    )

    local_pool_indices = np.argpartition(
        -priority[eligible_positions],
        kth=pool_size - 1
    )[:pool_size]

    candidate_pool = eligible_positions[
        local_pool_indices
    ]

    candidate_pool = candidate_pool[
        np.argsort(
            -priority[candidate_pool]
        )
    ]

    available = set(candidate_pool.tolist())
    uncovered_priority = priority.copy()

    selected_positions = []

    for _ in range(maximum_patrols):
        if not available:
            break

        available_array = np.fromiter(
            available,
            dtype=np.int32
        )

        direct_scores = priority[
            available_array
        ]

        marginal_coverage_scores = np.asarray(
            coverage_matrix[
                available_array
            ].dot(uncovered_priority)
        ).ravel()

        direct_max = direct_scores.max()
        coverage_max = (
            marginal_coverage_scores.max()
        )

        direct_normalized = np.divide(
            direct_scores,
            direct_max,
            out=np.zeros_like(
                direct_scores,
                dtype=np.float32
            ),
            where=direct_max > 0
        )

        coverage_normalized = np.divide(
            marginal_coverage_scores,
            coverage_max,
            out=np.zeros_like(
                marginal_coverage_scores,
                dtype=np.float32
            ),
            where=coverage_max > 0
        )

        utility = (
            BEST_BALANCE_ALPHA
            * direct_normalized
            + (
                1 - BEST_BALANCE_ALPHA
            )
            * coverage_normalized
        )

        best_position = int(
            available_array[
                np.argmax(utility)
            ]
        )

        selected_positions.append(
            best_position
        )

        available.remove(best_position)

        covered_positions = (
            coverage_matrix[
                best_position
            ].indices
        )

        uncovered_priority[
            covered_positions
        ] = 0

    return selected_positions


adaptive_selection_rows = []

for hour_position in range(N_HOURS):
    selected_positions = (
        adaptive_balanced_patrol_order(
            priority=priority_matrix[
                hour_position
            ],
            alpha=BEST_BALANCE_ALPHA,
            minimum_score=MIN_DISPATCH_SCORE,
            candidate_pool_size=100,
            maximum_patrols=25
        )
    )

    for selection_order, zone_position in enumerate(
        selected_positions,
        start=1
    ):
        adaptive_selection_rows.append({
            "target_time":
                target_time_order[
                    hour_position
                ],
            "split_code":
                split_by_hour[
                    hour_position
                ],
            "selection_order":
                selection_order,
            "zone_index":
                zone_order[
                    zone_position
                ],
            "zone_position":
                zone_position,
            "spatial_priority_score":
                priority_matrix[
                    hour_position,
                    zone_position
                ]
        })

adaptive_patrol_selections = pd.DataFrame(
    adaptive_selection_rows
)

adaptive_selection_lookup = {
    target_time: group.sort_values(
        "selection_order"
    )["zone_position"].to_numpy(
        dtype=np.int32
    )
    for target_time, group in (
        adaptive_patrol_selections.groupby(
            "target_time"
        )
    )
}

adaptive_hour_counts = (
    adaptive_patrol_selections.groupby(
        "target_time"
    ).size()
    .reindex(
        target_time_order,
        fill_value=0
    )
)

print(
    "Adaptive selection rows:",
    len(adaptive_patrol_selections)
)

display(
    adaptive_hour_counts.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    ).to_frame(
        "patrols_available_per_hour"
    )
)

In [ ]:
# Compare fixed-capacity and adaptive deployment

# This evaluates how much pressure and hotspot coverage is retained when low-risk recommendations are removed

def evaluate_selection_lookup(
    selection_lookup_to_use,
    hour_positions,
    k
):
    direct_pressure = 0.0
    area_pressure = 0.0
    total_pressure = 0.0

    direct_hotspots = 0
    area_hotspots = 0
    total_hotspots = 0

    deployed_counts = []
    overlap_rates = []

    for hour_position in hour_positions:
        target_time = target_time_order[
            hour_position
        ]

        selected_positions = (
            selection_lookup_to_use.get(
                target_time,
                np.empty(
                    0,
                    dtype=np.int32
                )
            )[:k]
        )

        deployed_counts.append(
            len(selected_positions)
        )

        actual = actual_matrix[
            hour_position
        ]

        covered = np.zeros(
            N_ZONES,
            dtype=bool
        )

        individual_coverage_total = 0

        for selected_position in (
            selected_positions
        ):
            covered_positions = (
                coverage_matrix[
                    selected_position
                ].indices
            )

            covered[
                covered_positions
            ] = True

            individual_coverage_total += len(
                covered_positions
            )

        direct_pressure += actual[
            selected_positions
        ].sum()

        area_pressure += actual[
            covered
        ].sum()

        total_pressure += actual.sum()

        hotspot_mask = actual >= 8

        direct_hotspots += hotspot_mask[
            selected_positions
        ].sum()

        area_hotspots += hotspot_mask[
            covered
        ].sum()

        total_hotspots += hotspot_mask.sum()

        overlap_rates.append(
            (
                1
                - covered.sum()
                / individual_coverage_total
            )
            if individual_coverage_total > 0
            else 0
        )

    return {
        "k_capacity": k,
        "mean_patrols_deployed":
            np.mean(deployed_counts),
        "median_patrols_deployed":
            np.median(deployed_counts),
        "zero_patrol_hour_percent":
            100
            * np.mean(
                np.asarray(deployed_counts)
                == 0
            ),
        "full_capacity_hour_percent":
            100
            * np.mean(
                np.asarray(deployed_counts)
                == k
            ),
        "direct_pressure_capture":
            direct_pressure
            / total_pressure,
        "area_pressure_coverage":
            area_pressure
            / total_pressure,
        "direct_hotspot_recall":
            (
                direct_hotspots
                / total_hotspots
                if total_hotspots > 0
                else 0
            ),
        "area_hotspot_coverage":
            (
                area_hotspots
                / total_hotspots
                if total_hotspots > 0
                else 0
            ),
        "mean_overlap_rate":
            np.mean(overlap_rates)
    }


fixed_selection_lookup = {
    target_time: positions
    for (
        alpha,
        target_time
    ), positions in selection_lookup.items()
    if alpha == BEST_BALANCE_ALPHA
}

adaptive_evaluation_rows = []

evaluation_periods = {
    "validation_holdout":
        validation_hour_positions,
    "test":
        test_hour_positions
}

for period_name, hour_positions in (
    evaluation_periods.items()
):
    for strategy_name, lookup in [
        (
            "fixed_capacity",
            fixed_selection_lookup
        ),
        (
            "adaptive_dispatch",
            adaptive_selection_lookup
        )
    ]:
        for k in [5, 10, 25]:
            row = evaluate_selection_lookup(
                selection_lookup_to_use=lookup,
                hour_positions=hour_positions,
                k=k
            )

            row["period"] = period_name
            row["strategy"] = strategy_name

            adaptive_evaluation_rows.append(
                row
            )

adaptive_dispatch_evaluation = pd.DataFrame(
    adaptive_evaluation_rows
)

display(
    adaptive_dispatch_evaluation[
        [
            "period",
            "strategy",
            "k_capacity",
            "mean_patrols_deployed",
            "zero_patrol_hour_percent",
            "full_capacity_hour_percent",
            "direct_pressure_capture",
            "area_pressure_coverage",
            "direct_hotspot_recall",
            "area_hotspot_coverage",
            "mean_overlap_rate"
        ]
    ].sort_values(
        [
            "period",
            "k_capacity",
            "strategy"
        ]
    )
)

In [ ]:
# save adaptive dispatch evidence
dispatch_threshold_summary.to_csv(
    OUTPUT_DIR
    / "adaptive_dispatch_threshold.csv",
    index=False
)

adaptive_patrol_selections.to_parquet(
    OUTPUT_DIR
    / "adaptive_patrol_selections.parquet",
    index=False
)

adaptive_dispatch_evaluation.to_csv(
    OUTPUT_DIR
    / "adaptive_dispatch_evaluation.csv",
    index=False
)

print("Adaptive patrol outputs saved.")

In [ ]:
adaptive_selection_base = adaptive_patrol_selections[
    [
        "zone_index",
        "target_time",
        "split_code",
        "selection_order",
        "zone_position"
    ]
].copy()

adaptive_recommendation_columns = [
    "zone_index",
    "target_time",
    "split_code",
    "time_window",
    "incident_count",
    "unique_vehicles",
    "predicted_pressure",
    "calibrated_pressure",
    "spatial_priority_score",
    "hotspot_probability",
    "distance_neighbour_exposure",
    "active_neighbour_count",
    "active_neighbour_fraction",
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share",
    "explicit_road_obstruction",
    "large_vehicle_evidence",
    "multi_offence_evidence",
    "context_evidence_breadth",
    "disruption_class_code",
    "disruption_class",
    "observation_confidence",
    "confidence_tier",
    "observer_bias_penalty",
    "duplicate_capture_rate",
    "cold_start_zone",
    "current_burst_z",
    "relative_to_same_hour",
    "short_term_trend",
    "consecutive_active_hours_before",
    "recommended_response",
    "local_pressure_removed_30pct",
    "local_pressure_removed_60pct",
    "local_pressure_removed_90pct",
    "actual_pressure"
]

adaptive_recommendation_columns = [
    col for col in adaptive_recommendation_columns
    if col in operational_panel.columns
]

adaptive_recommendations = (
    adaptive_selection_base.merge(
        operational_panel[
            adaptive_recommendation_columns
        ],
        on=[
            "zone_index",
            "target_time",
            "split_code"
        ],
        how="left",
        validate="one_to_one"
    )
)

spatial_columns = [
    "zone_index",
    "zone_250m",
    "latitude",
    "longitude",
    "centroid_x_m",
    "centroid_y_m"
]

spatial_columns = [
    col for col in spatial_columns
    if col in zone_lookup.columns
]

adaptive_recommendations = (
    adaptive_recommendations.merge(
        zone_lookup[spatial_columns],
        on="zone_index",
        how="left",
        validate="many_to_one"
    )
)

adaptive_recommendations["split"] = (
    adaptive_recommendations["split_code"]
    .map({
        2: "validation",
        3: "test"
    })
)


def create_adaptive_evidence_text(row):
    evidence = []

    if row["incident_count"] > 0:
        evidence.append(
            f"{int(row['incident_count'])} current incident(s)"
        )

    if row.get("current_burst_z", 0) >= 2:
        evidence.append(
            "unusual burst above recent baseline"
        )

    if row.get("relative_to_same_hour", 0) >= 2:
        evidence.append(
            "at least twice normal same-hour activity"
        )

    if row.get(
        "consecutive_active_hours_before",
        0
    ) >= 2:
        evidence.append(
            "persistent activity across recent hours"
        )

    if row.get("active_neighbour_count", 0) >= 2:
        evidence.append(
            f"{int(row['active_neighbour_count'])} nearby active zones"
        )

    if row.get(
        "explicit_road_obstruction",
        0
    ) == 1:
        evidence.append(
            "road-interface obstruction evidence"
        )

    if row.get(
        "large_vehicle_evidence",
        0
    ) == 1:
        evidence.append(
            "large-vehicle involvement"
        )

    if row.get(
        "multi_offence_evidence",
        0
    ) == 1:
        evidence.append(
            "multiple-offence evidence"
        )

    if not evidence:
        evidence.append(
            "forecast recurrence without current contextual evidence"
        )

    return "; ".join(evidence)


adaptive_recommendations["evidence_summary"] = (
    adaptive_recommendations.apply(
        create_adaptive_evidence_text,
        axis=1
    )
)

adaptive_recommendations["patrol_action"] = np.select(
    [
        adaptive_recommendations[
            "confidence_tier"
        ].eq("low_confidence"),

        (
            adaptive_recommendations[
                "incident_count"
            ].gt(0)
            & adaptive_recommendations[
                "disruption_class_code"
            ].eq(2)
        ),

        adaptive_recommendations[
            "incident_count"
        ].gt(0)
    ],
    [
        "verify_before_enforcement",
        "targeted_obstruction_enforcement",
        "active_parking_enforcement"
    ],
    default="preventive_patrol"
)

adaptive_recommendations["confidence_note"] = np.select(
    [
        adaptive_recommendations[
            "confidence_tier"
        ].eq("low_confidence"),

        adaptive_recommendations[
            "confidence_tier"
        ].eq("high_confidence")
    ],
    [
        "limited observation reliability; verify on site",
        "strong historical observation support"
    ],
    default="moderate observation support"
)

assert len(adaptive_recommendations) == len(
    adaptive_patrol_selections
)

assert not adaptive_recommendations.duplicated(
    ["target_time", "zone_index"]
).any()

assert adaptive_recommendations[
    "spatial_priority_score"
].ge(MIN_DISPATCH_SCORE).all()


# Create one status row for every target hour, including zero-patrol hours.
adaptive_dispatch_schedule = (
    operational_panel[
        ["target_time", "split_code"]
    ]
    .drop_duplicates()
    .sort_values("target_time")
)

hourly_dispatch_counts = (
    adaptive_recommendations.groupby(
        "target_time"
    )
    .agg(
        recommended_patrols=(
            "zone_index",
            "size"
        ),
        maximum_priority_score=(
            "spatial_priority_score",
            "max"
        ),
        mean_selected_confidence=(
            "observation_confidence",
            "mean"
        )
    )
    .reset_index()
)

adaptive_dispatch_schedule = (
    adaptive_dispatch_schedule.merge(
        hourly_dispatch_counts,
        on="target_time",
        how="left",
        validate="one_to_one"
    )
)

adaptive_dispatch_schedule[
    "recommended_patrols"
] = (
    adaptive_dispatch_schedule[
        "recommended_patrols"
    ]
    .fillna(0)
    .astype("int16")
)

adaptive_dispatch_schedule[
    "deployment_status"
] = np.where(
    adaptive_dispatch_schedule[
        "recommended_patrols"
    ] > 0,
    "deployment_recommended",
    "no_deployment_recommended"
)

latest_target_time = (
    adaptive_dispatch_schedule[
        "target_time"
    ].max()
)

latest_adaptive_recommendations = (
    adaptive_recommendations.loc[
        adaptive_recommendations[
            "target_time"
        ] == latest_target_time
    ]
    .sort_values("selection_order")
    .reset_index(drop=True)
)

operational_adaptive_export = (
    adaptive_recommendations.drop(
        columns=["actual_pressure"],
        errors="ignore"
    )
)

operational_adaptive_export.to_parquet(
    OUTPUT_DIR
    / "final_adaptive_patrol_recommendations.parquet",
    index=False
)

adaptive_recommendations.to_parquet(
    OUTPUT_DIR
    / "final_adaptive_patrol_recommendations_with_evaluation.parquet",
    index=False
)

adaptive_dispatch_schedule.to_csv(
    OUTPUT_DIR
    / "adaptive_hourly_dispatch_schedule.csv",
    index=False
)

latest_adaptive_recommendations.drop(
    columns=["actual_pressure"],
    errors="ignore"
).to_csv(
    OUTPUT_DIR
    / "latest_adaptive_enforcement_recommendations.csv",
    index=False
)

print(
    "Adaptive recommendation rows:",
    len(adaptive_recommendations)
)

print(
    "Hours with no deployment:",
    int(
        adaptive_dispatch_schedule[
            "recommended_patrols"
        ].eq(0).sum()
    )
)

print(
    "Latest target hour:",
    latest_target_time
)

print(
    "Latest recommended patrols:",
    len(latest_adaptive_recommendations)
)

display(
    latest_adaptive_recommendations[
        [
            "selection_order",
            "zone_index",
            "zone_250m",
            "latitude",
            "longitude",
            "spatial_priority_score",
            "calibrated_pressure",
            "hotspot_probability",
            "disruption_class",
            "observation_confidence",
            "patrol_action",
            "local_pressure_removed_60pct",
            "evidence_summary"
        ]
    ]
)